In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from package.databases.initialize import initialize_memories

initialize_memories()

In [3]:
from dataclasses import dataclass, field
from typing import List, Optional
import pickle
from package.flows.offline.actions import Enrich, Jargon, Parallel
from package.flows.offline.flow import get_parallel_actions, Shared
from package.interface import SourceOptions
from package.utils.data_loder import PDFLoader
from package.databases.session import Session, get_session, Depends
from package.databases.management.document import DocumentManagement, Document
from package.databases.management.longterm import LongTermManagement, LongTerm

def create_document(source_path:str):
    dm = DocumentManagement()
    ltm = LongTermManagement()
    source_type = source_path.split(".")[-1]

    source_ops = SourceOptions(
        path=source_path,
        type=source_type if source_type in ['pdf'] else "other"
    )
    document = Document(source=source_path, type=source_type)
    document = dm.create_document(document, session=Depends(get_session))

    contexts = PDFLoader(source=source_ops).run()
    longterms = [LongTerm(document_id=document.id, raw=context.context, meta=context.metadata) for context in contexts]
    ltm.create_raws(longterms, session=Depends(get_session))

    _longterms = ltm.read_longterms_by_document(document_id=document.id, session=Depends(get_session))
    longterms = []
    for l in _longterms:
        _l = l.__dict__.copy()
        if hasattr(_l, '_sa_instance_state'):
            delattr(_l, '_sa_instance_state')
        longterms.append(LongTerm(**_l))

    return document, longterms

@dataclass
class OfflineIndexRunner:
    content_path: str
    enrich_system_prompt: str = './package/flows/offline/prompts/enricher.md'
    term_system_prompt: str = './package/flows/offline/prompts/term_extractor.md'
    term_model:str = 'us.meta.llama4-maverick-17b-instruct-v1:0'
    chunks: List[LongTerm] = field(default_factory=list)
    current_chunk_index: int = 0
    completed_count: int = 0
    state_path: str = './offline_index_state.pkl'
    document_id: Optional[str] = None
    document_source: Optional[str] = None
    
    def load_content(self):
        document, longterms = create_document(self.content_path)
        self.document_id = document.id
        self.document_source = document.source
        self.chunks.extend(longterms)

    def run(self):
        if len(self.chunks)==0:
            self.load_content()
        try:
            for i in range(self.current_chunk_index, len(self.chunks)):
                self.current_chunk_index = i
                chunk = self.chunks[i]
                
                # Process single chunk: Parallel(Enrich, Jargon) + Embed
                shared = Shared(
                    enrich_system_prompt=self.enrich_system_prompt,
                    term_system_prompt=self.term_system_prompt,
                    term_model=self.term_model,
                    chunk=chunk
                )
                flow = get_parallel_actions()
                flow.run(shared)
                self.completed_count += 1
                
        except KeyboardInterrupt:
            print("Interrupted by user. Saving state...")
            self.save_state()
            print(f"State saved. Completed {self.completed_count} tasks.")
            raise                
        except Exception as e:
            print("Error:", str(e))
            self.save_state()
            raise  

    def save_state(self):
        with open(self.state_path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load_state(cls):
        try:
            with open(cls.state_path, 'rb') as f:
                return pickle.load(f)
        except FileNotFoundError as e:
            print("Error:", str(e))
            return None

d:\broai-arai\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
runner = OfflineIndexRunner(
    content_path='./sources/Assisting in Writing Wikipedia-like Articles From Scratch with Large Language Models.pdf',
)

In [5]:
runner.run()

d:\broai-arai\backend\package\utils\data_loder.py:22: UserWarning: [EXPERIMENT] You're using an experimental module, which is subject to change in future.: split_markdown
  chunks = split_markdown(text)
d:\broai-arai\backend\package\utils\data_loder.py:23: UserWarning: [EXPERIMENT] You're using an experimental module, which is subject to change in future.: consolidate_markdown
  consolidated_chunks = consolidate_markdown(chunks)
d:\broai-arai\backend\package\utils\data_loder.py:24: UserWarning: [EXPERIMENT] You're using an experimental module, which is subject to change in future.: get_markdown_sections
  sections = get_markdown_sections(consolidated_chunks)
d:\broai-arai\backend\package\utils\data_loder.py:30: UserWarning: [EXPERIMENT] You're using an experimental module, which is subject to change in future.: split_overlap
  new_contexts = split_overlap(contexts, max_tokens=max_tokens, overlap=overlap)


Markdown headings: max(2)


In [6]:
from package.databases.management.longterm import LongTermManagement, LongTerm
from package.databases.management.term import TermManagement, Term
from package.databases.management.document import DocumentManagement, Document

In [7]:
tm = TermManagement()
dm = DocumentManagement()
ltm = LongTermManagement()

In [8]:
dm.read_documents(session=Depends(get_session))

[Document(id='cf80c844-5632-4384-b75a-50e3f64f30a0', status=<DocumentStatus.PENDING: 'pending'>, updated_at=datetime.datetime(2025, 8, 5, 19, 16, 17, 25884), source='./sources/Assisting in Writing Wikipedia-like Articles From Scratch with Large Language Models.pdf', type='pdf', created_at=datetime.datetime(2025, 8, 5, 19, 16, 17, 25884))]

In [24]:
target = "STORM"
terms = tm.read_terms(session=Depends(get_session))
for i in terms:
    if target.lower() == i.term.lower():
        if target.lower() != i.evidence.lower().strip():
            print("{term} | {type} | {evidence} | {explanation}".format(term=i.term, type=i.type, evidence=i.evidence, explanation=i.explanation))
            print("="*20)

STORM | Acronym | STORM , a writing system for the S ynthesis of T opic O utlines through R etrieval and M ulti-perspective Question Asking | STORM stands for Synthesis of Topic Outlines through Retrieval and Multi-perspective Question Asking
STORM | Acronym | we propose the STORM paradigm for the S ynthesis | STORM is a paradigm proposed for Synthesis
STORM | Acronym | STORM paradigm for the S ynthesis of T opic O utlines through R etrieval and M ulti-perspective Question Asking | STORM is a paradigm for the synthesis of topic outlines through retrieval and multi-perspective question asking
STORM | Technical Term | We present STORM to automate the pre-writing stage | STORM is a system that automates the pre-writing stage
STORM | Proper Name | STORM discovers different perspectives by surveying existing articles from similar topics | STORM is a system or method that discovers different perspectives on a topic
STORM | Proper Name | STORM simulates a conversation | STORM is a system that

In [10]:
dm.read_documents(session=Depends(get_session))

[Document(id='cf80c844-5632-4384-b75a-50e3f64f30a0', status=<DocumentStatus.PENDING: 'pending'>, updated_at=datetime.datetime(2025, 8, 5, 19, 16, 17, 25884), source='./sources/Assisting in Writing Wikipedia-like Articles From Scratch with Large Language Models.pdf', type='pdf', created_at=datetime.datetime(2025, 8, 5, 19, 16, 17, 25884))]

In [12]:
longterms = ltm.read_longterms(session=Depends(get_session))
len(longterms)

87